In [1]:
import pandas as pd
import numpy as np
import re
pd.reset_option('display.max_colwidth')

In [2]:
dt = pd.read_csv("combined_dataset.csv", encoding='latin1')
dt.head()

,Source.Name,ACN,Date,Aircraft Make Model,Event Label,Contributing Factors,Primary Problem,Report 1,Report 2
0,ASRS_DBOnline.csv,1319608,201512,B737 Undifferentiated or Other Model,Aircraft Equipment Problem Less Severe,Aircraft,Aircraft,Level at FL320 simultaneously the autopilot ki...,[Report narrative contained no additional info...
1,ASRS_DBOnline.csv,1319659,201512,B747-400,Deviation - Speed All Types; Deviation / Discr...,Human Factors,Human Factors,We were on a 25 mile downwind. We were at 170 ...,Aircraft did what it was commanded by FMC and ...
2,ASRS_DBOnline.csv,1319720,201512,B777-200,Aircraft Equipment Problem Less Severe; Deviat...,Environment - Non Weather Related; Weather; Ai...,Ambiguous,We [were] flying a polar route and [were] not ...,Returning from break; FB and FC reported to me...
3,ASRS_DBOnline.csv,1319791,201512,B737-800,Aircraft Equipment Problem Critical; Flight De...,Aircraft,Aircraft,Prior to flight; Crew Oxygen system gauge read...,NaN
4,ASRS_DBOnline.csv,1319852,201512,A300,Aircraft Equipment Problem Critical,Aircraft,Aircraft,As throttles retarded at top of descent; numbe...,[Report narrative contained no additional info...


In [3]:
dt['Event Label'].nunique()

3409

In [4]:
dt['Event Label'] = dt['Event Label'].str.split(';').str[0].str.strip()
print(dt['Event Label'].nunique())
dt['Event Label'].value_counts()


56


Event Label
Aircraft Equipment Problem Critical                                  4523
Aircraft Equipment Problem Less Severe                               4148
ATC Issue All Types                                                  1836
Deviation / Discrepancy - Procedural Published Material / Policy     1732
Conflict Ground Conflict                                              539
Deviation / Discrepancy - Procedural Hazardous Material Violation     405
Deviation / Discrepancy - Procedural FAR                              378
Deviation - Altitude Excursion From Assigned Altitude                 264
Deviation - Speed All Types                                           262
Deviation - Track / Heading All Types                                 236
Inflight Event / Encounter Wake Vortex Encounter                      186
Flight Deck / Cabin / Aircraft Event Smoke / Fire / Fumes / Odor      161
Flight Deck / Cabin / Aircraft Event Illness / Injury                 155
Conflict Airborne Conflict

In [5]:
counts = dt['Event Label'].value_counts()
dt=dt[dt['Event Label']!="No Specific Anomaly Occurred All Types"]
dt = dt[dt['Event Label'].isin(counts[counts > 2].index)]
dt['Event Label'].value_counts()


Event Label
Aircraft Equipment Problem Critical                                  4523
Aircraft Equipment Problem Less Severe                               4148
ATC Issue All Types                                                  1836
Deviation / Discrepancy - Procedural Published Material / Policy     1732
Conflict Ground Conflict                                              539
Deviation / Discrepancy - Procedural Hazardous Material Violation     405
Deviation / Discrepancy - Procedural FAR                              378
Deviation - Altitude Excursion From Assigned Altitude                 264
Deviation - Speed All Types                                           262
Deviation - Track / Heading All Types                                 236
Inflight Event / Encounter Wake Vortex Encounter                      186
Flight Deck / Cabin / Aircraft Event Smoke / Fire / Fumes / Odor      161
Flight Deck / Cabin / Aircraft Event Illness / Injury                 155
Conflict Airborne Conflict

In [6]:
# Consolidation of labels into 6 main categories/labels
def consolidate_label(label):
    if 'Aircraft Equipment Problem' in label:
        return 'Equipment Problem'
    elif 'Deviation' in label:
        return 'Deviation'
    elif 'ATC Issue' in label:
        return 'ATC Issue'
    elif 'Conflict' in label:
        return 'Conflict'
    elif 'Inflight Event' in label:
        return 'Inflight Event'
    elif 'Ground Event' in label or 'Ground Excursion' in label:
        return 'Ground Event'
    else:
        return None

dt['Event Label'] = dt['Event Label'].apply(consolidate_label)
dt = dt[dt['Event Label'].notna()]
dt['Event Label'].value_counts()

Event Label
Equipment Problem    8671
Deviation            3925
ATC Issue            1836
Conflict              822
Inflight Event        410
Ground Event          229
Name: count, dtype: int64

In [7]:
report1_null_count=dt['Report 1'].isna().sum()
report2_null_count=dt['Report 2'].isna().sum()

print((dt['Report 1'] == '[Report narrative contained no additional information.]').sum())
print((dt['Report 2'] == '[Report narrative contained no additional information.]').sum())

print(report1_null_count)
print(report2_null_count)


0
1057
0
10702


In [8]:
placeholder = '[Report narrative contained no additional information.]'
dt['Report 2'] = dt['Report 2'].replace(placeholder, '')
dt["Narrative Text"]=dt['Report 1'].fillna("") + " " + dt['Report 2'].fillna("")
df=dt.drop(columns=['Report 1','Report 2'])


In [9]:
# Main dataset (df). dt- Testing and cleaning dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15893 entries, 0 to 16344
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Source.Name           15893 non-null  object
 1   ACN                   15893 non-null  int64 
 2   Date                  15893 non-null  int64 
 3   Aircraft Make Model   15893 non-null  object
 4   Event Label           15893 non-null  object
 5   Contributing Factors  15871 non-null  object
 6   Primary Problem       15871 non-null  object
 7   Narrative Text        15893 non-null  object
dtypes: int64(2), object(6)
memory usage: 1.1+ MB


Text PreProcessing

In [ ]:
df['Narrative Text'] = df['Narrative Text'].str.lower()
df['Narrative Text'] = df['Narrative Text'].str.replace(r'[^\w\s]', '', regex=True)
